In [84]:
SEED=47
root_dataset_dir = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/participant_swow_collection_us"

In [85]:
import os
os.environ['workspaceFolder'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms"
os.environ['WORKING_DIR'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms"
os.environ['PYTHONPATH'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms"

os.environ['HF_HOME'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/../huggingface"
os.environ['TRITON_CACHE_DIR'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/../triton_cache"

In [86]:
import pandas as pd

cue_meta_info = pd.read_json(f'/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/participant_swow_collection_us/participant_swow_collection_us_seed_{SEED}/cue_meta_info.jsonl')

In [87]:
cue_meta_info

,seed,cue_info
25%,46,"[Cue word: rejection, Cue word: punk, Cue word..."
50%,46,"[Cue word: deserving, Cue word: bong, Cue word..."
75%,46,"[Cue word: psych, Cue word: associated, Cue wo..."
100%,46,"[Cue word: hearsay, Cue word: lifetime, Cue wo..."
test,46,"[Cue word: cruelty, Cue word: sentimental, Cue..."


In [88]:
print(len(cue_meta_info.iloc[0]['cue_info']), '25%')
print(len(cue_meta_info.iloc[1]['cue_info']), '50%')

2456 25%
2456 50%


In [89]:
# pick 5% and 10% cues 
ten_percent_in_25 = 0.1 / 0.25
five_percent_in_10 = 0.05 / 0.10


In [90]:
# pick 10% cues 
from random import sample
ten_percent_random_indices = sample(range(len(cue_meta_info.iloc[0]['cue_info'])), int(len(cue_meta_info.iloc[0]['cue_info']) * ten_percent_in_25))
# sort the indices
ten_percent_random_indices.sort()
ten_percent_cues = [cue_meta_info.iloc[0]['cue_info'][i] for i in ten_percent_random_indices]

In [91]:
# pick 5% cues 
five_percent_random_indices = sample(range(len(ten_percent_cues)), int(len(ten_percent_cues) * five_percent_in_10))
# sort the indices
five_percent_random_indices.sort()
five_percent_cues = [ten_percent_cues[i] for i in five_percent_random_indices]

In [92]:
print(f"len of 10% cues: {len(ten_percent_cues)}")
print(f"len of 5% cues: {len(five_percent_cues)}")

len of 10% cues: 982
len of 5% cues: 491


In [93]:
five_percent_cues[0]

'Cue word: encounter'

In [94]:
import datasets
# load dataset 

participant_swow_collection = datasets.load_dataset(root_dataset_dir,f"participant_swow_collection_us_seed_{SEED}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [95]:
participant_swow_collection['train'][0]

{'system': 'You are a human participant in a psychology experiment.',
 'instruction': '### Background ###\nOn average, an adult knows about 40,000 words, but what do these words mean to people like you and me? You can help scientists understand how meaning is organized in our mental dictionary by playing the game of word associations. This game is easy: Just give the first three words that come to mind for a given cue word.\n### OUTPUT FORMAT ###\nOutput your response in the following format:\nresponse1, response2, response3\nDo not provide any additional context, or explanations. Just the words as comma-separated values.\nIf you cannot think of further responses after response1 or response2, output the token NO MORE RESPONSE for the remaining slot(s).\n### End of instructions ###\n',
 'input': 'Cue word: mace',
 'output': 'spice, pepper spray, medieval weapon'}

In [96]:
# loop over participant_swow_collection['train'], collect data whose 'input' key is in five_percent_cues
from tqdm.auto import tqdm
ten_percent_dataset = []
for data in tqdm(participant_swow_collection['train']):
    if data['input'] in ten_percent_cues:
        ten_percent_dataset.append(data)

  0%|          | 0/957619 [00:00<?, ?it/s]

In [97]:
# get five directly from ten_percent_dataset
five_percent_dataset = []
for data in tqdm(ten_percent_dataset):
    if data['input'] in five_percent_cues:
        five_percent_dataset.append(data)
        
print(f"len of 10% dataset: {len(ten_percent_dataset)}")
print(f"len of 5% dataset: {len(five_percent_dataset)}")

  0%|          | 0/95524 [00:00<?, ?it/s]

len of 10% dataset: 95524
len of 5% dataset: 47692


In [98]:
five_percent_dataset[0]

{'system': 'You are a human participant in a psychology experiment.',
 'instruction': '### Background ###\nOn average, an adult knows about 40,000 words, but what do these words mean to people like you and me? You can help scientists understand how meaning is organized in our mental dictionary by playing the game of word associations. This game is easy: Just give the first three words that come to mind for a given cue word.\n### OUTPUT FORMAT ###\nOutput your response in the following format:\nresponse1, response2, response3\nDo not provide any additional context, or explanations. Just the words as comma-separated values.\nIf you cannot think of further responses after response1 or response2, output the token NO MORE RESPONSE for the remaining slot(s).\n### End of instructions ###\n',
 'input': 'Cue word: genetics',
 'output': 'history, family, genes'}

In [83]:
from pathlib import Path
five_save_dataset_name = f'isolated_5_percent_participant_swow_collection_us_seed_{SEED}'
full_five_save_pf = os.path.join(root_dataset_dir, five_save_dataset_name, 'train', 'chunk_0.jsonl')
Path(os.path.dirname(full_five_save_pf)).mkdir(parents=True, exist_ok=True)

ten_save_dataset_name = f'isolated_10_percent_participant_swow_collection_us_seed_{SEED}'
full_ten_save_pf = os.path.join(root_dataset_dir, ten_save_dataset_name, 'train', 'chunk_0.jsonl')
Path(os.path.dirname(full_ten_save_pf)).mkdir(parents=True, exist_ok=True)
import jsonlines

with jsonlines.open(full_five_save_pf, mode='w') as writer:
    for data in five_percent_dataset:
        writer.write(data)
with jsonlines.open(full_ten_save_pf, mode='w') as writer:
    for data in ten_percent_dataset:
        writer.write(data)

In [35]:
# ! finally  
# # try to load the saved dataset
five_dataset = datasets.load_dataset(root_dataset_dir,f"isolated_5_percent_participant_swow_collection_us_seed_{SEED}")
ten_dataset = datasets.load_dataset(root_dataset_dir,f"isolated_10_percent_participant_swow_collection_us_seed_{SEED}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]